# Project 08: Google Play Store Market Analytics & Consumer Sentiment Engine

**Author:** Dhruv Goyal ([GitHub](https://github.com/DhruvGoyal-create))  
**Track:** Data Analytics & Business Intelligence  
**Specialization:** Empirical Market Intelligence & NLP  

---

## 1. Executive Summary & Objective
This notebook presents an end-to-end empirical market analysis of the Google Play Store software ecosystem across **10,841 application listings** and **64,295 consumer review records**.

### Objectives:
1. **Category Saturation:** Diagnose catalog distribution across 33 app verticals.
2. **Rating Quality Benchmarks:** Measure central tendencies, skewness, and category variances.
3. **Package Size Elasticity:** Analyze Spearman correlation between APK size (MB) and install tiers.
4. **Monetization Mechanics:** Evaluate free vs. paid software and upfront revenue distributions.
5. **Consumer Sentiment Polarity:** Process user review text using NLP (TextBlob) to map satisfaction.

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import core analytical pipeline functions
from google_play_store_analysis import load_and_clean_data, generate_tables, generate_figures

base_dir = os.getcwd()
print(f"Working Directory: {base_dir}")

Working Directory: C:\Users\HP\OneDrive\Working on\Prj Dev\Prj Work Space 4(DS & AIML)\Primary A1-Data Science\Secondary A2-Bussiness Analytics\projects\08_Google_Play_Store_Analytics


## 2. Ingestion & Data Hygiene Protocol
- Handles corrupted **Row 10472** (Life Made WI-Fi Touchscreen Photo Frame) where missing category shifted columns (Rating = 19.0).
- Strips `+` and `,` from `Installs`, `$` from `Price`.
- Parses alphanumeric `Size` to float `Size_MB`.
- Deduplicates snapshot scrape redundancies by keeping the latest record per app (sorted by `Reviews` descending).

In [2]:
df_apps_raw, df_apps_clean, df_rev_raw, df_rev_clean, df_merged = load_and_clean_data(base_dir)

print(f"Raw Apps Count: {len(df_apps_raw):,} | Clean Unique Apps: {len(df_apps_clean):,}")
print(f"Raw Reviews Count: {len(df_rev_raw):,} | Clean Reviews with Text: {len(df_rev_clean):,}")
print(f"Merged Dataset Size: {len(df_merged):,} across {df_merged['Category'].nunique()} categories.")
df_apps_clean.head(5)

Raw Apps Count: 10,841 | Clean Unique Apps: 9,659
Raw Reviews Count: 64,295 | Clean Reviews with Text: 37,427


Merged Dataset Size: 35,929 across 33 categories.


,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver,Installs_Numeric,Price_Numeric,Size_MB,Est_Min_Upfront_Revenue
2544,Facebook,SOCIAL,4.1,78158306,Varies with device,"1,000,000,000+",Free,0,Teen,Social,"August 3, 2018",Varies with device,Varies with device,1000000000,0.0,NaN,0.0
381,WhatsApp Messenger,COMMUNICATION,4.4,69119316,Varies with device,"1,000,000,000+",Free,0,Everyone,Communication,"August 3, 2018",Varies with device,Varies with device,1000000000,0.0,NaN,0.0
2604,Instagram,SOCIAL,4.5,66577446,Varies with device,"1,000,000,000+",Free,0,Teen,Social,"July 31, 2018",Varies with device,Varies with device,1000000000,0.0,NaN,0.0
382,Messenger – Text and Video Chat for Free,COMMUNICATION,4.0,56646578,Varies with device,"1,000,000,000+",Free,0,Everyone,Communication,"August 1, 2018",Varies with device,Varies with device,1000000000,0.0,NaN,0.0
1879,Clash of Clans,GAME,4.6,44893888,98M,"100,000,000+",Free,0,Everyone 10+,Strategy,"July 15, 2018",10.322.16,4.1 and up,100000000,0.0,98.0,0.0


## 3. Market Saturation & Category Distribution
Evaluating catalog breadth across top Google Play Store categories.

In [3]:
cat_summary = pd.read_csv('outputs/tables/category_saturation_analysis.csv')
cat_summary.head(10)

,Category,App_Count,Total_Installs,Median_Installs,Mean_Rating,Median_Rating,App_Share_Pct,Install_Share_Pct
0,FAMILY,1877,6236542505,50000.0,4.183898,4.3,19.432653,8.279784
1,GAME,945,13447924415,1000000.0,4.244432,4.3,9.783621,17.853788
2,TOOLS,829,8102771915,50000.0,4.040278,4.2,8.582669,10.757435
3,BUSINESS,420,697164865,1000.0,4.098479,4.2,4.348276,0.925573
4,MEDICAL,395,38293077,1000.0,4.165862,4.3,4.089450,0.050839
5,PERSONALIZATION,376,1532494782,10000.0,4.332215,4.4,3.892743,2.034577
6,PRODUCTIVITY,374,5793091369,100000.0,4.183389,4.3,3.872036,7.691048
7,LIFESTYLE,369,503823539,10000.0,4.093355,4.2,3.820271,0.668888
8,FINANCE,345,455348734,10000.0,4.115563,4.3,3.571798,0.604532
9,SPORTS,325,1096474498,100000.0,4.216154,4.3,3.364738,1.455706


## 4. User Rating Distribution & Central Tendencies
Quantifying consumer review metrics and variance across verticals.

In [4]:
ratings_summary = pd.read_csv('outputs/tables/category_ratings_summary.csv')
ratings_summary.head(10)

,Category,Count,Mean,Std,Median,Min,Max,KW_H_Statistic,KW_P_Value
0,EVENTS,45,4.435556,0.419499,4.5,2.9,5.0,237.9375,2.543700e-33
1,ART_AND_DESIGN,59,4.359322,0.359654,4.4,3.2,5.0,237.9375,2.543700e-33
2,EDUCATION,105,4.351429,0.266788,4.4,3.5,4.9,237.9375,2.543700e-33
3,BOOKS_AND_REFERENCE,169,4.344970,0.437649,4.5,2.7,5.0,237.9375,2.543700e-33
4,PERSONALIZATION,298,4.332215,0.358968,4.4,2.5,5.0,237.9375,2.543700e-33
5,PARENTING,50,4.300000,0.517845,4.4,2.0,5.0,237.9375,2.543700e-33
6,BEAUTY,42,4.278571,0.362603,4.3,3.1,4.9,237.9375,2.543700e-33
7,SOCIAL,203,4.247291,0.457209,4.3,1.9,5.0,237.9375,2.543700e-33
8,GAME,898,4.244432,0.385636,4.3,1.0,5.0,237.9375,2.543700e-33
9,WEATHER,72,4.243056,0.337692,4.3,3.3,4.8,237.9375,2.543700e-33


## 5. Technical Package Size (MB) vs. Download Elasticity
Spearman rank correlation coefficients measuring download sensitivity to application binary footprint.

In [5]:
corr_summary = pd.read_csv('outputs/tables/size_install_correlation_stats.csv')
corr_summary

,Size_Tier,App_Count,Median_Installs,Mean_Installs,Mean_Rating,Spearman_Rho,Spearman_P,Pearson_Log_R,Pearson_P
0,< 10 MB,3930,10000.0,1.119563e+06,4.131254,0.3101,2.128300e-187,0.2998,1.260600e-174
1,10 - 25 MB,2101,50000.0,4.164672e+06,4.182979,0.3101,2.128300e-187,0.2998,1.260600e-174
2,25 - 50 MB,1524,100000.0,4.539229e+06,4.144436,0.3101,2.128300e-187,0.2998,1.260600e-174
3,50 - 100 MB,876,1000000.0,1.300742e+07,4.250903,0.3101,2.128300e-187,0.2998,1.260600e-174


## 6. Commercial Monetization & Upfront Revenue Estimates
Free vs. Paid adoption distributions and lower-bound upfront revenue benchmarks.

In [6]:
rev_summary = pd.read_csv('outputs/tables/category_revenue_estimates.csv')
rev_summary.head(10)

,Category,Total_Apps,Paid_Apps,Free_Apps,Avg_Paid_Price,Max_Paid_Price,Total_Paid_Installs,Est_Min_Upfront_Revenue
0,FAMILY,1877,185,1692,13.026162,399.99,21150814,1.156808e+08
1,LIFESTYLE,369,19,350,124.256316,400.00,1179110,5.758394e+07
2,GAME,945,82,863,3.467195,17.99,20999965,4.068864e+07
3,FINANCE,345,17,328,170.637059,399.99,185602,2.572668e+07
4,PHOTOGRAPHY,281,19,262,6.225263,29.99,1878740,8.547768e+06
5,PERSONALIZATION,376,81,295,1.857778,9.99,3258794,6.796948e+06
6,MEDICAL,395,82,313,11.984268,200.00,560733,5.956237e+06
7,TOOLS,829,78,751,3.426282,25.99,1727441,5.464821e+06
8,SPORTS,325,24,301,4.166667,29.99,1243815,4.706212e+06
9,PRODUCTIVITY,374,28,346,8.961786,154.99,1412055,4.313375e+06


## 7. Natural Language Sentiment & Review Polarity Analysis
Qualitative sentiment distribution and polarity scores across software categories.

In [7]:
sent_summary = pd.read_csv('outputs/tables/sentiment_classification_summary.csv')
sent_summary

,Sentiment_Class,Review_Count,Percentage,Mean_Polarity,Mean_Subjectivity
0,Positive,23998,64.119486,0.372402,0.566949
1,Negative,8271,22.099019,-0.256173,0.535087
2,Neutral,5158,13.781495,0.000000,0.079786


## 8. Summary & Strategic Recommendations
1. **Avoid Saturated Utility Red Oceans:** Domains like `TOOLS` and `FAMILY` exhibit massive catalog competition. Focus on niche spaces like `HEALTH_AND_FITNESS` and `EDUCATION`.
2. **Size Optimization:** Restrict utility app footprints below 25 MB; allow heavy asset budgets primarily for gaming where consumers expect rich 3D rendering.
3. **Monetization Selection:** Freemium/IAP remains essential for mass consumer applications, with direct upfront paywalls viable only for niche professional/medical software.